# 实时验配界面 —— 4 段增益

板子上另开一条程序在跑：**麦克风 → 核 → 耳机**，实时。
这一页**不参与音频**，它只和那条程序交换两个小文件：

| 文件 | 谁写 | 谁读 |
|---|---|---|
| `/dev/shm/fir_tune.json` | 这一页（4 根滑杆拧出来的值） | `fir_live.py` |
| `/dev/shm/fir_status.json` | `fir_live.py`（增益回显 + 电平） | 这一页 |

**所以：这一页关掉、卡住、刷新，声音都不会断。** 反过来也一样 —— 实时那条没起，
这里只会显示"读不到状态"，不会报错。

## 怎么用

1. 戴好耳机。四段插头要**插到底**；插着却录不到声音时见
   `skill/pitfalls/pynq_audio_playback.md` 坑 5 —— **要拔插到电平表跳起来为止**。
2. 从上往下依次运行本页所有单元（菜单 `Run → Run All Cells`）。
3. 点 **① 启动实时**。
4. 拖滑杆 —— 耳机里的声音应该跟着变。

## 这个界面在演示里做的是什么

助听器验配就是这个动作：**哪一段听不见，就把哪一段抬起来**。
听力正常的人四根滑杆都在 0 dB；把低频压下去、高频抬起来，
就是一张"高频听力损失"的听力图。点几个预设能直接听出区别。


In [ ]:
import json
import math
import os
import signal
import subprocess
import sys
import threading
import time

import ipywidgets as W
from IPython.display import display

HERE = os.getcwd()
if HERE not in sys.path:
    sys.path.insert(0, HERE)
from fir_tune import read_status, write_tune, TUNE_PATH, STATUS_PATH

print("ipywidgets", W.__version__)
print("旋钮文件", TUNE_PATH)
print("状态文件", STATUS_PATH)
print("→ 现在读不到状态：实时那条还没起（很正常，下一步点「启动」）"
      if read_status() is None else "→ 状态文件有了，实时那条在跑或跑过一轮")


In [ ]:
# ---------------- 4 根滑杆 ----------------
# 分段点来自核的源码（相减式互补 FIR）：b1 = 0~500，b2 = 500~1k，b3 = 1k~2k，b4 = 2k~24k。
BANDS = ["① 0 – 500 Hz", "② 500 – 1 kHz", "③ 1 – 2 kHz", "④ 2 – 24 kHz"]
GAIN_UI_LIMIT = 24.0        # 界面只给到 ±24 dB（核能到 ±42，演示用不了那么狠）

sld = [W.FloatSlider(value=0.0, min=-GAIN_UI_LIMIT, max=GAIN_UI_LIMIT, step=0.5,
                     description=b, continuous_update=True, readout_format="+.1f",
                     style={"description_width": "110px"},
                     layout=W.Layout(width="560px"))
       for b in BANDS]

PRESETS = {
    "平坦：不动（0 dB）":              (0, 0, 0, 0),
    "高频下降：典型老年性听力图":      (-4, -2, 6, 14),
    "只抬辅音：清辅音都在 2 kHz 以上": (-10, -2, 6, 10),
    "压掉低频轰鸣":                    (-16, -6, 2, 4),
    "极端对照：一拖就听出区别":        (-20, -10, 10, 20),
}
PLACEHOLDER = "（选一个套用）"
drop = W.Dropdown(options=[PLACEHOLDER] + list(PRESETS), value=PLACEHOLDER,
                  description="预设", style={"description_width": "60px"},
                  layout=W.Layout(width="430px"))
src = W.Dropdown(options=[("麦克风", "mic"), ("放 wav（没插麦也能验流水线）", "wav")],
                 value="mic", description="音源",
                 style={"description_width": "60px"})
byp = W.ToggleButton(value=False, description="直通（跳过压缩）",
                     layout=W.Layout(width="200px"))


# ---------------- 拧滑杆 → 写旋钮文件 ----------------
# **不是直接把参数据写进核** —— 这边一个寄存器都不碰。
# 实时那条自己在下一块开头把文件读走，所以这一页卡住也不会让音频卡住。
seq = [0]


def push_tune(*_):
    seq[0] += 1
    write_tune({"seq": seq[0],
                "gain_db": [round(s.value, 2) for s in sld],
                "bypass": 1 if byp.value else 0})


for _s in sld:
    _s.observe(push_tune, names="value")
byp.observe(push_tune, names="value")


def on_preset(change):
    if change["new"] in PRESETS:
        for _s, _v in zip(sld, PRESETS[change["new"]]):
            _s.value = _v              # 每改一根都会自己写一次旋钮文件，多写几次不要紧
        drop.value = PLACEHOLDER       # 拨回提示项，同一个预设才能再点一次


drop.observe(on_preset, names="value")


# ---------------- 起停实时那条 ----------------
PROC = {"p": None, "log": None}


def start(_):
    p = PROC["p"]
    if p is not None and p.poll() is None:
        print("已经在跑了（pid %d）" % p.pid)
        return
    env = dict(os.environ)
    env.update({"XILINX_XRT": "/usr", "PYTHONUNBUFFERED": "1",
                "LIVE_SRC": src.value, "LIVE_SECONDS": "0",
                "LIVE_GAIN_DB": ",".join("%.2f" % s.value for s in sld)})
    if PROC["log"]:
        PROC["log"].close()
    PROC["log"] = open("/tmp/fir_live_ui.log", "wb")
    PROC["p"] = subprocess.Popen(
        [sys.executable, os.path.join(HERE, "fir_live.py")], cwd=HERE, env=env,
        stdout=PROC["log"], stderr=subprocess.STDOUT, start_new_session=True)
    print("起来了，pid %d，音源 %s" % (PROC["p"].pid, src.value))
    # 起跑检查。两种最常见的失败都发生在头一两秒：已经有一条在跑了（fir_live.py
    # 自己用文件锁挡住了），或者加载 overlay 失败。两种情况它都会马上退出，
    # 原因写在日志里 —— 直接翻出来，别让人对着一动不动的数字猜。
    time.sleep(2.0)
    rc = PROC["p"].poll()
    if rc is not None:
        print("\n它起跑就退了（退出码 %s）。日志最后几行：" % rc)
        print("  " + open("/tmp/fir_live_ui.log", encoding="utf-8",
                          errors="replace").read()[-1200:].strip())
    else:
        print("日志 /tmp/fir_live_ui.log")


def stop(_):
    p = PROC["p"]
    if p is None or p.poll() is not None:
        print("没在跑")
        return
    # 发 SIGINT 而不是 SIGTERM：SIGINT 走 KeyboardInterrupt 那条路，
    # 收尾会把音频通路正常关掉（那一下静音）。SIGTERM 是直接打死，耳机里会留一声。
    p.send_signal(signal.SIGINT)
    try:
        p.wait(timeout=8)
        print("停了（pid %d）" % p.pid)
    except subprocess.TimeoutExpired:
        p.kill()
        print("它没理 SIGINT，强杀了 —— 收尾那一下静音可能没做")


btn_run = W.Button(description="① 启动实时", button_style="success")
btn_stop = W.Button(description="② 停止", button_style="warning")
btn_run.on_click(start)
btn_stop.on_click(stop)

push_tune()      # 先把当前滑杆位置写进去，这样"先拧好再启动"也生效
display(W.VBox([W.HTML("<b>每段增益</b> —— 拖一下，耳机里应该马上跟着变"),
                *sld, W.HBox([drop, src]), W.HBox([byp, btn_run, btn_stop])]))


In [ ]:
# ---------------- 表针（后台线程，10 Hz） ----------------
# 为什么用后台线程：在单元里写 `while True:` 会把内核占死，滑块全不响应。
meter = W.HTML()
readout = W.HTML()
WATCH = {"stop": threading.Event(), "th": None}


def db_of(rms):
    """int16 域有效值 → dBFS。满量程 32768 = 0 dBFS。"""
    return 20.0 * math.log10(rms / 32768.0) if rms and rms > 0 else -120.0


def lvl_bar(db, width=200):
    pct = max(0.0, min(100.0, (db + 60.0) / 60.0 * 100.0))
    col = "#2ecc71" if db < -12 else ("#f1c40f" if db < -3 else "#e74c3c")
    return ('<span style="display:inline-block;width:%dpx;height:12px;'
            'background:#e8e8e8;border-radius:2px;vertical-align:middle">'
            '<span style="display:inline-block;width:%.0f%%;height:12px;'
            'background:%s;border-radius:2px"></span></span>' % (width, pct, col))


def gain_bar(db, lim=GAIN_UI_LIMIT):
    """中间一条零线：往右是抬、往左是压。"""
    px = max(0.0, min(1.0, abs(db) / lim)) * 55.0
    col = "#2ecc71" if db >= 0 else "#3498db"
    if db >= 0:
        seg = ('<span style="position:absolute;left:55px;width:%.1fpx;height:12px;'
               'background:%s"></span>' % (px, col))
    else:
        seg = ('<span style="position:absolute;left:%.1fpx;width:%.1fpx;height:12px;'
               'background:%s"></span>' % (55.0 - px, px, col))
    return ('<span style="position:relative;display:inline-block;width:110px;'
            'height:12px;background:#e8e8e8;border-radius:2px;vertical-align:middle">'
            '<span style="position:absolute;left:55px;width:1px;height:12px;'
            'background:#999"></span>%s</span>' % seg)


def _watch():
    while not WATCH["stop"].is_set():
        st = read_status()
        if st is None:
            meter.value = ('<span style="color:#b00">读不到 %s</span> —— '
                           '实时那条还没起？' % STATUS_PATH)
            readout.value = ""
            time.sleep(0.3)
            continue

        app = st.get("gain_applied_db") or [None] * 4
        tgt = st.get("gain_db") or [None] * 4
        rows = []
        for _i, _name in enumerate(BANDS):
            a = 0.0 if app[_i] is None else app[_i]
            t = 0.0 if tgt[_i] is None else tgt[_i]
            rows.append(
                '<tr><td style="padding:2px 10px 2px 0">%s</td>'
                '<td style="padding-right:8px"><b>%+.1f dB</b></td>'
                '<td style="padding-right:8px;color:#888;font-size:12px">'
                '目标 %+.1f</td><td>%s</td></tr>' % (_name, a, t, gain_bar(a)))

        db_in, db_out = db_of(st.get("in_rms")), db_of(st.get("out_rms"))
        act = (db_out - db_in) if (st.get("in_rms") or 0) > 0 else 0.0
        meter.value = (
            '<table style="font-size:13px">%s</table>'
            '<div style="margin-top:8px;font-size:13px">'
            '进核 %s %.1f dBFS&nbsp;&nbsp;&nbsp; 出核 %s %.1f dBFS'
            '&nbsp;&nbsp;&nbsp; 压缩动作 <b>%+.1f dB</b></div>'
            % ("".join(rows), lvl_bar(db_in), db_in, lvl_bar(db_out), db_out, act))

        # 帧有多旧？running=True 但帧很旧 = 那条进程卡住了或者被 kill -9 了。
        # 没有这一条的话，界面会把最后一帧当成"现在"一直显示下去 —— 最坑的假象。
        age = time.time() - (st.get("t") or 0)
        if not st.get("running"):
            tail = ("已停止，自检通过" if st.get("ok")
                    else ("已停止，自检<b style='color:#b00'>没过</b>"
                          if st.get("ok") is False else "已停止"))
        elif age > 1.5:
            tail = ("<b style='color:#b00'>说是运行中，但状态已经 %.1f 秒没更新了 —— "
                    "下面的数字是旧的</b>" % age)
        else:
            tail = "运行中"
        if st.get("error"):
            tail += " <span style='color:#b00'>报错：%s</span>" % st["error"]
        readout.value = (
            '<div style="font-size:13px">第 %s 块&nbsp;&nbsp; 实时 %s'
            '&nbsp;&nbsp; 丢块 %s&nbsp;&nbsp; 搬样 %s ms&nbsp;&nbsp;'
            ' 过核 %s ms&nbsp;&nbsp; %s</div>' % (
                st.get("n", 0),
                ("%.2fx" % st["rt"]) if st.get("rt") else "—",
                st.get("lost_blocks", 0),
                ("%.2f" % st["io_ms"]) if st.get("io_ms") else "—",
                ("%.2f" % st["dsp_ms"]) if st.get("dsp_ms") else "—",
                tail))
        time.sleep(0.1)


# 重跑本单元时先把上一个表针线程停掉 —— 不然两个线程一起往同一个 meter 里写。
WATCH["stop"].set()
if WATCH["th"] is not None:
    WATCH["th"].join(timeout=1.0)
WATCH["stop"].clear()
WATCH["th"] = threading.Thread(target=_watch, daemon=True)
WATCH["th"].start()

display(meter, readout)


## 数字怎么看

| 数字 | 正常范围 | 不对时先查什么 |
|---|---|---|
| 进核 dBFS | 说话时 **−30 ～ −15** | < −50 = 麦克风没进信号。见坑 5：**要拔插到电平表跳起来为止**（实测第 35.5 秒才跳） |
| 实时倍率 | **≥ 1.00x** | 掉到 1 以下而且丢块在涨：搬样没跟上 |
| 丢块 | 每 30 秒 **个位数** | 一直涨 → 停掉重开；还涨就把 `LIVE_L` 调到 960 |
| 压缩动作 | **−4 dB 上下** | 接近 0：进核电平太低，压缩器根本没启动（它只在超过门限时才动） |

## "起跑就退了"

**同时只能有一条 `fir_live.py`。** 两条一起跑会互相踩 MMIO，而且会**同时往
同一个状态文件里写** —— 界面上的块号会来回跳，看着像"通路在抽搐"，其实是
两个进程在抢着汇报。所以 `fir_live.py` 自己用文件锁挡住了第二条。

点「启动」之后如果它两秒内就退了，脚本会把日志最后几行打出来，照那个改。
手工清场：`pkill -INT -f fir_live.py`。

## 两处"看着像坏了，其实没坏"

- **回显比滑杆慢半拍、还差零点几 dB。** 故意的：增益分 4 块（40 ms）线性走到位，
  直接跳变会在耳机里"啪"一声；差的那点小数是 Q7.8 的量化（一步 0.034 dB）。
- **拖滑杆时声音是"滑"过去的，不是瞬间换。** 同上。

## 这一页对应 PLAN §11 的哪一条

§11.2 第 ③ 条：4 条滑杆 + 预设 + 读数回显 + 电平条。
**"导入听力图"**（从一张 audiogram 自动算出每段增益）是下一步；
卡点 ①（10/21）之前做不完就砍掉它，保住滑杆。
